# 4.2 Pixels + similarité cosinus : Dataset 2

Même méthode et mêmes 5 représentations qu'en 3.2. Comparées sur le 1er extrait (30 000 images),
puis les pixels bruts sont vérifiés sur l'extrait final.

In [1]:
import cv2
import numpy as np
import pandas as pd
from skimage.feature import hog
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from outils_4 import list_images, classify_by_cosine

DOSSIER_30K = "../data/rvl_cdip_filtered_dataset_2"
DOSSIER_60K = "../data/rvl_cdip_filtered_dataset_3"

## 1. Calcul des représentations sur les 30 000 images (environ 5 minutes)

In [2]:
df = list_images(DOSSIER_30K)
y = df["label"].values

raw, profils, hogs, formes = [], [], [], []
for chemin in df["filepath"]:
    image = cv2.imread(chemin, cv2.IMREAD_GRAYSCALE)
    hauteur, largeur = image.shape
    # otsu : 1 = encre, 0 = fond
    _, encre = cv2.threshold(image, 0, 1, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # raw : l'image en 64x64, mise à plat
    raw.append(cv2.resize(image, (64, 64), interpolation=cv2.INTER_AREA).ravel() / 255)

    # profiles : quantité d'encre par ligne et par colonne
    grille = np.linspace(0, 1, 256)
    par_ligne = np.interp(grille, np.linspace(0, 1, hauteur), encre.mean(axis=1))
    par_colonne = np.interp(grille, np.linspace(0, 1, largeur), encre.mean(axis=0))
    profils.append(np.concatenate([par_ligne, par_colonne]))

    # hog : orientation des contours
    hogs.append(hog(cv2.resize(image, (128, 128), interpolation=cv2.INTER_AREA),
                    orientations=9, pixels_per_cell=(16, 16), cells_per_block=(3, 3)))

    # shape : juste le format et la quantité d'encre
    formes.append([hauteur / largeur, encre.mean()])

# on découpe les numéros de ligne, pas les images
idx_train, idx_test = train_test_split(np.arange(len(df)), test_size=0.2, random_state=1, stratify=y)

# float32 pour ne pas saturer la mémoire avec 30 000 images
raw = np.array(raw, dtype=np.float32)
representations = {
    "raw": raw,
    "centered": raw - raw[idx_train].mean(axis=0),  # moyenne du train seulement
    "profiles": np.array(profils, dtype=np.float32),
    "hog": np.array(hogs, dtype=np.float32),
    "shape": np.array(formes, dtype=np.float32),
}
print(f"{len(df)} images | Train : {len(idx_train)} | Test : {len(idx_test)}")

30000 images | Train : 24000 | Test : 6000


## 2. Centroïdes + cosinus pour chaque représentation

In [3]:
lignes = []
for nom, X in representations.items():
    prediction, similarites = classify_by_cosine(X[idx_train], y[idx_train], X[idx_test])

    # marge = ressemblance à sa propre classe - ressemblance aux autres
    est_vraie_classe = np.array([similarites.columns == vrai for vrai in y[idx_test]])
    intra = similarites.values[est_vraie_classe].mean()
    inter = similarites.values[~est_vraie_classe].mean()

    lignes.append({"représentation": nom, "accuracy": accuracy_score(y[idx_test], prediction),
                   "sim_intra": intra, "sim_inter": inter, "marge": intra - inter})

pd.DataFrame(lignes).sort_values("accuracy", ascending=False).round(4)

,représentation,accuracy,sim_intra,sim_inter,marge
3,hog,0.5063,0.7728,0.7179,0.0549
0,raw,0.4435,0.9923,0.9920,0.0003
2,profiles,0.4297,0.6158,0.5779,0.0379
1,centered,0.3633,0.1540,-0.0752,0.2292
4,shape,0.2748,0.9991,0.9990,0.0002


`hog` est la meilleure, à peine au-dessus des 6 mesures de 4.1. `raw` : plus de 0,99 de similarité
partout, le cosinus ne sépare presque rien.

## 3. Contrôle sur l'extrait final (60 000 images) : pixels bruts (environ 4 minutes)

In [4]:
df_60k = list_images(DOSSIER_60K)
y_60k = df_60k["label"].values
raw_60k = np.array([cv2.resize(cv2.imread(c, cv2.IMREAD_GRAYSCALE), (64, 64), interpolation=cv2.INTER_AREA).ravel() / 255
                    for c in df_60k["filepath"]], dtype=np.float32)

idx_train, idx_test = train_test_split(np.arange(len(df_60k)), test_size=0.2, random_state=42, stratify=y_60k)
prediction, similarites = classify_by_cosine(raw_60k[idx_train], y_60k[idx_train], raw_60k[idx_test])
est_vraie_classe = np.array([similarites.columns == vrai for vrai in y_60k[idx_test]])
intra = similarites.values[est_vraie_classe].mean()
inter = similarites.values[~est_vraie_classe].mean()

print(f"{len(df_60k)} images | Test : {len(idx_test)}")
print(f"Accuracy : {accuracy_score(y_60k[idx_test], prediction):.3f}")
print(f"sim_intra = {intra:.4f} | sim_inter = {inter:.4f} | marge = {intra - inter:.4f}")

60000 images | Test : 12000
Accuracy : 0.447
sim_intra = 0.9915 | sim_inter = 0.9912 | marge = 0.0003
